# Plant Disease Classification using CNN

A smart agriculture company wants to help farmers detect plant diseases early using AI. Build an image classification model that identifies the disease affecting a tomato plant from a photo of its leaf, so farmers can act before it spreads.

You are given real leaf photographs organized into folders, one folder per class. Build a complete image classification pipeline and train a CNN of your own design that can tell the 6 classes apart.

## Dataset

| Item | Details |
|------|---------|
| Archive | `plant_disease_dataset.zip` (the setup cell extracts it to `dataset/`) |
| Structure | `dataset/<ClassName>/*.jpg` (the folder name is the label) |
| Images | Real leaf photographs |
| Total images | 1500 (250 per class) |
| Classes (6) | `Early_Blight`, `Healthy`, `Late_Blight`, `Leaf_Mold`, `Septoria_Leaf_Spot`, `Yellow_Curl_Virus` |

## What You Need to Build

1. **Explore the data.** Gather the image paths and their labels (a pandas DataFrame is a convenient way to do this) and look at a few sample images from each class.

2. **Encode the labels.** Map the class names to integer labels 0 to 5 in alphabetical order:

   | Class | Label |
   |-------|-------|
   | `Early_Blight` | 0 |
   | `Healthy` | 1 |
   | `Late_Blight` | 2 |
   | `Leaf_Mold` | 3 |
   | `Septoria_Leaf_Spot` | 4 |
   | `Yellow_Curl_Virus` | 5 |

   Your model's outputs must follow this ordering so the evaluation can score predictions correctly.

3. **Prepare the data.** Hold out part of the data for validation (an 80/20 split works well). Build a PyTorch `Dataset` and create `DataLoader` objects for training and validation.

4. **Build a CNN.** Design your own convolutional network that takes input of shape (N, 3, 64, 64) and outputs scores for the 6 classes. The depth, layer types, activations, and regularization are your choice.

5. **Train it.** Train your model so it generalizes well to leaves it has not seen. The optimizer, number of epochs, batch size, and other training details are up to you.

6. **Evaluate it.** Report the validation accuracy, display a confusion matrix over the full validation set, and show the predicted vs actual labels for a few validation images.

## Evaluation

Your model is tested on a hidden set of leaf images that are loaded as RGB, resized to 64 x 64, scaled to [0, 1] (divide by 255), and arranged as (N, 3, 64, 64). Prepare your training data the same way so your model performs well at test time.

To pass, your trained model should reach at least 60% accuracy on the hidden test images.

## Notes

* The setup cell extracts the dataset and imports common libraries for you.
* The confusion matrix should be computed on your full validation set.
* Make sure the notebook runs end to end without errors and finishes within the time limit.
* Do not use plt.show() in your final submission.


In [1]:
# Run this cell before writing your solution
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

if not os.path.isdir("dataset"):
    with zipfile.ZipFile("plant_disease_dataset.zip", "r") as zip_ref:
        zip_ref.extractall(".")

print(os.listdir("dataset"))


['Late_Blight', 'Yellow_Curl_Virus', 'Septoria_Leaf_Spot', 'Leaf_Mold', 'Early_Blight', 'Healthy']


1. Explore the data

In [2]:
data_dir = 'dataset'
classes = sorted(os.listdir(data_dir))
print('Classes:', classes)

image_paths = []
labels = []

for label_name in classes:
  class_folder = os.path.join(data_dir, label_name)
  if os.path.isdir(class_folder):
    for img_name in os.listdir(class_folder):
      if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        image_paths.append(os.path.join(class_folder, img_name))
        labels.append(label_name)

df = pd.DataFrame({'image_path': image_paths, 'class_name': labels})
print(df.head())
print(df['class_name'].value_counts())

Classes: ['Early_Blight', 'Healthy', 'Late_Blight', 'Leaf_Mold', 'Septoria_Leaf_Spot', 'Yellow_Curl_Virus']
                                  image_path    class_name
0  dataset/Early_Blight/early_blight_187.jpg  Early_Blight
1  dataset/Early_Blight/early_blight_069.jpg  Early_Blight
2  dataset/Early_Blight/early_blight_237.jpg  Early_Blight
3  dataset/Early_Blight/early_blight_198.jpg  Early_Blight
4  dataset/Early_Blight/early_blight_220.jpg  Early_Blight
class_name
Early_Blight          250
Healthy               250
Late_Blight           250
Leaf_Mold             250
Septoria_Leaf_Spot    250
Yellow_Curl_Virus     250
Name: count, dtype: int64


2. Encode the labels.

In [3]:
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
print('Class to Index Mapping:', class_to_idx)

df['label'] = df['class_name'].map(class_to_idx)
print(df.head())

Class to Index Mapping: {'Early_Blight': 0, 'Healthy': 1, 'Late_Blight': 2, 'Leaf_Mold': 3, 'Septoria_Leaf_Spot': 4, 'Yellow_Curl_Virus': 5}
                                  image_path    class_name  label
0  dataset/Early_Blight/early_blight_187.jpg  Early_Blight      0
1  dataset/Early_Blight/early_blight_069.jpg  Early_Blight      0
2  dataset/Early_Blight/early_blight_237.jpg  Early_Blight      0
3  dataset/Early_Blight/early_blight_198.jpg  Early_Blight      0
4  dataset/Early_Blight/early_blight_220.jpg  Early_Blight      0


3. Prepare the data.

In [4]:
from torchvision import transforms

transform = transforms.Compose([
  transforms.Resize((64, 64)),
  transforms.ToTensor(),
])

class PlantDataset(Dataset):
  def __init__(self, dataframe, transform=None):
    self.df = dataframe.reset_index(drop=True)
    self.transform = transform

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    img_path = self.df.loc[idx, 'image_path']
    image = Image.open(img_path).convert('RGB')
    label = self.df.loc[idx, 'label']

    if self.transform:
      image = self.transform(image)

    return image, torch.tensor(label, dtype=torch.long)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_dataset = PlantDataset(train_df, transform=transform)
val_dataset = PlantDataset(val_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

4. Build a CNN.

In [5]:
class PlantCNN(nn.Module):
  def __init__(self, num_classes=6):
    super(PlantCNN, self).__init__()
    self.features = nn.Sequential(
      nn.Conv2d(3, 32, kernel_size=3, padding=1),
      nn.BatchNorm2d(32),
      nn.ReLU(),
      nn.MaxPool2d(2, 2),

      nn.Conv2d(32, 64, kernel_size=3, padding=1),
      nn.BatchNorm2d(64),
      nn.ReLU(),
      nn.MaxPool2d(2, 2),

      nn.Conv2d(64, 128, kernel_size=3, padding=1),
      nn.BatchNorm2d(128),
      nn.ReLU(),
      nn.MaxPool2d(2, 2)
    )

    self.classifier = nn.Sequential(
      nn.Flatten(),
      nn.Linear(128 * 8 * 8, 256),
      nn.ReLU(),
      nn.Dropout(0.5),
      nn.Linear(256, num_classes)
    )

  def forward(self, x):
    x =self.features(x)
    x = self.classifier(x)
    return x 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PlantCNN(num_classes=6).to(device)
print(model)

PlantCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=8192, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout

5. Train it.

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs=10

for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0 
  correct = 0 
  total = 0 

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()  * images.size(0)
    _, predicted = outputs.max(1)
    total += labels.size(0)
    correct += predicted.eq(labels).sum().item()

  epoch_loss = running_loss / total 
  epoc_acc = correct / total
  print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoc_acc:.4f}')

Epoch [1/10], Loss: 2.1191, Accuracy: 0.4000
Epoch [2/10], Loss: 1.0617, Accuracy: 0.6067
Epoch [3/10], Loss: 0.9376, Accuracy: 0.6517
Epoch [4/10], Loss: 0.7817, Accuracy: 0.7033
Epoch [5/10], Loss: 0.6626, Accuracy: 0.7408
Epoch [6/10], Loss: 0.5866, Accuracy: 0.7817
Epoch [7/10], Loss: 0.5457, Accuracy: 0.8058
Epoch [8/10], Loss: 0.4947, Accuracy: 0.8100
Epoch [9/10], Loss: 0.4538, Accuracy: 0.8208
Epoch [10/10], Loss: 0.4151, Accuracy: 0.8608


6. Evaluate it.

In [8]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
  for images, labels in val_loader:
    images = images.to(device)
    outputs = model(images)
    _, predicted = outputs.max(1)

    all_preds.extend(predicted.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

val_accuracy = accuracy_score(all_labels, all_preds)
print(f'Validation Accuracy: {val_accuracy:.4f}')

cm = confusion_matrix(all_labels, all_preds)
print('Confusion Matrix')
print(cm)

idx_to_class = {v: k for k, v in class_to_idx.items()}
print('\nSample Predicitons vs Actual:')
for i in range(5):
  img, label = val_dataset[i]
  pred_label = all_preds[i]
  print(f'Sample {i+1}: True = {idx_to_class[label.item()]} | Predicted = {idx_to_class[pred_label]}')

Validation Accuracy: 0.7400
Confusion Matrix
[[27  0  7  1 12  3]
 [ 4 40  0  1  5  0]
 [ 6  0 33  0 10  1]
 [ 0  0  0 30 20  0]
 [ 0  0  0  2 48  0]
 [ 3  0  0  0  3 44]]

Sample Predicitons vs Actual:
Sample 1: True = Late_Blight | Predicted = Late_Blight
Sample 2: True = Early_Blight | Predicted = Early_Blight
Sample 3: True = Early_Blight | Predicted = Early_Blight
Sample 4: True = Leaf_Mold | Predicted = Leaf_Mold
Sample 5: True = Yellow_Curl_Virus | Predicted = Yellow_Curl_Virus
